# Module 04: CPU Instruction Timing & Memory Capacity
**AE2.021 · USTH**

This notebook computes instruction execution time from T-states/clock frequency, computes required
memory capacity from the number of DRAM chips, and simulates a simple fetch-decode-execute cycle.

## 1. Instruction execution time: t = n_T × (1/f_clk)
🎯 **What question does this method answer?** Knowing the clock frequency and the number of T-states an instruction needs, how to derive the exact real execution time in seconds: and re-check the original question (50 MHz, 11 T-states).

In [1]:
def exec_time(n_tstates: int, f_clk_hz: float) -> float:
    T = 1.0 / f_clk_hz
    return n_tstates * T

# Original question (Tooley Ch.7 Q15): 50MHz, 11 T-states -> expected 220ns
t = exec_time(11, 50e6)
print(f"50MHz, 11 T-state -> t = {t*1e9:.1f} ns  (expected 220 ns)")

# Generated question: 20MHz, 4 T-states -> expected 200ns
t2 = exec_time(4, 20e6)
print(f"20MHz, 4 T-state  -> t = {t2*1e9:.1f} ns  (expected 200 ns)")

print()
for f_mhz in (10, 20, 50, 100):
    print(f"f_clk={f_mhz:3d} MHz -> 1 T-state = {1/(f_mhz*1e6)*1e9:.1f} ns")

50MHz, 11 T-state -> t = 220.0 ns  (expected 220 ns)
20MHz, 4 T-state  -> t = 200.0 ns  (expected 200 ns)

f_clk= 10 MHz -> 1 T-state = 100.0 ns
f_clk= 20 MHz -> 1 T-state = 50.0 ns
f_clk= 50 MHz -> 1 T-state = 20.0 ns
f_clk=100 MHz -> 1 T-state = 10.0 ns


#### 📤 Real output
The two computed results (220 ns and 200 ns) match exactly the answers published on the website for the original question and the corresponding generated question: confirming that the formula t = n_T × (1/f_clk) and its application are correct.

## 2. Memory capacity from the number of DRAM chips
🎯 **What question does this method answer?** Given N bytes to store and each DRAM chip holding only (rows×cols) bits, how many chips are needed at minimum: and how many memory cells can be addressed with a given n-bit address bus.

In [2]:
def chips_needed(total_bytes: int, chip_words: int, chip_bits_per_word: int) -> int:
    chip_bytes = chip_words * chip_bits_per_word // 8
    return -(-total_bytes // chip_bytes)  # round up

# Original question (Tooley Ch.6 Q13): how many 16K x 4-bit ICs are needed for 32K bytes?
n_chips = chips_needed(32*1024, 16*1024, 4)
print(f"32KB from 16Kx4bit ICs -> needs {n_chips} ICs (expected 4)")

def addressable_locations(address_bus_bits: int) -> int:
    return 2 ** address_bus_bits

for bits in (8, 16, 24, 32):
    n = addressable_locations(bits)
    print(f"Address bus {bits:2d}-bit -> can address {n:>12,} cells "
          f"(largest hex = {n-1:X})")

32KB from 16Kx4bit ICs -> needs 4 ICs (expected 4)
Address bus  8-bit -> can address          256 cells (largest hex = FF)
Address bus 16-bit -> can address       65,536 cells (largest hex = FFFF)
Address bus 24-bit -> can address   16,777,216 cells (largest hex = FFFFFF)
Address bus 32-bit -> can address 4,294,967,296 cells (largest hex = FFFFFFFF)


#### 📤 Real output
`chips_needed` gives exactly 4: matching the original Tooley Ch.6 Q13 question. With a 24-bit bus, the largest hex address is FFFFFF: matching the original Ch.6 Q5 question. With a 32-bit bus, the number of addressable cells is 4,294,967,296 = 4 GB: matching the added generated question.

## 3. A simple simulation of the fetch–decode–execute cycle
🎯 **What question does this method answer?** A concrete illustration of each step: the PC points to the instruction, the instruction is loaded into the IR, decoded, then executed: instead of only describing it in words.

In [3]:
program = {
    0: ("LOAD", "A", 5),   # A = 5
    1: ("LOAD", "B", 3),   # B = 3
    2: ("ADD",  "A", "B"), # A = A + B
    3: ("HALT", None, None),
}

registers = {"A": 0, "B": 0}
PC = 0

def fetch(pc):
    return program[pc]

def execute(instr):
    op, x, y = instr
    if op == "LOAD":
        registers[x] = y
    elif op == "ADD":
        registers[x] = registers[x] + registers[y]
    elif op == "HALT":
        return False
    return True

running = True
while running:
    instr = fetch(PC)               # FETCH
    print(f"PC={PC}  IR={instr}   (fetch)")
    op = instr[0]                   # DECODE (simplified here: read the opcode directly)
    running = execute(instr)        # EXECUTE
    print(f"        -> after execution: registers={registers}")
    PC += 1

print("\nFinal result:", registers)

PC=0  IR=('LOAD', 'A', 5)   (fetch)
        -> after execution: registers={'A': 5, 'B': 0}
PC=1  IR=('LOAD', 'B', 3)   (fetch)
        -> after execution: registers={'A': 5, 'B': 3}
PC=2  IR=('ADD', 'A', 'B')   (fetch)
        -> after execution: registers={'A': 8, 'B': 3}
PC=3  IR=('HALT', None, None)   (fetch)
        -> after execution: registers={'A': 8, 'B': 3}

Final result: {'A': 8, 'B': 3}


#### 📤 Real output
After 4 fetch-decode-execute cycles, register A = 8 (5+3), as the defined program says: a visual illustration of the PC increasing step by step, each instruction being fetched before it is executed, in the order of the instruction cycle taught in the theory part.

In [4]:
print("\n" + "="*70, "\nFloyd Ch.11 - Section 11-1 Semiconductor Memory Basics (Problem 4)\n" + "="*70)
print("4) Memory address (0-256) corresponding to each hex number:")
for h in ["0C","5E","DF"]:
    print(f"   {h}16 = {int(h,16)} (decimal)")

print("\n" + "="*70, "\nFloyd Ch.11 - Problem 2 (concept, no calculation): relation between bit/byte/nibble/word\n" + "="*70)
print("   1 nibble = 4 bit. 1 byte = 8 bit = 2 nibble. 1 word = a multiple of a byte (usually 16/32/64-bit = 2/4/8 bytes),")
print("   the word width depends on the specific CPU architecture.")

print("\n" + "="*70, "\nFloyd Ch.11 - Section 11-1 Problem 5 (static RAM with 4 rows, given conditions)\n" + "="*70)
rows = [0,0,0,0]
conditions = [(0,1,1),(1,0,1),(2,1,0),(3,0,1)]  # (row_index, row_select, data_in_bit)
print("   Initially all 4 rows = 0. Apply the conditions (write only when Row select=1):")
for row_idx, sel, data in conditions:
    if sel == 1:
        rows[row_idx] = data
        print(f"   Row {row_idx} selected (select=1), writes Data in={data} -> Row {row_idx} = {data}")
    else:
        print(f"   Row {row_idx} NOT selected (select=0) -> unchanged = {rows[row_idx]}")
print("   Final contents of the 4 rows:", rows)



Floyd Ch.11 - Section 11-1 Semiconductor Memory Basics (Problem 4)
4) Memory address (0-256) corresponding to each hex number:
   0C16 = 12 (decimal)
   5E16 = 94 (decimal)
   DF16 = 223 (decimal)

Floyd Ch.11 - Problem 2 (concept, no calculation): relation between bit/byte/nibble/word
   1 nibble = 4 bit. 1 byte = 8 bit = 2 nibble. 1 word = a multiple of a byte (usually 16/32/64-bit = 2/4/8 bytes),
   the word width depends on the specific CPU architecture.

Floyd Ch.11 - Section 11-1 Problem 5 (static RAM with 4 rows, given conditions)
   Initially all 4 rows = 0. Apply the conditions (write only when Row select=1):
   Row 0 selected (select=1), writes Data in=1 -> Row 0 = 1
   Row 1 NOT selected (select=0) -> unchanged = 0
   Row 2 selected (select=1), writes Data in=0 -> Row 2 = 0
   Row 3 NOT selected (select=0) -> unchanged = 0
   Final contents of the 4 rows: [1, 0, 0, 0]
